# 03. Advanced Nonlinear Ensemble Modeling
> **Random Forest & XGBoost optimization with NLP synopsis embeddings**

Part of the **GreenlightML** open-source film risk modeling and portfolio optimization engine.


## 1. Setup & Advanced Feature Encoding
Incorporate NLP synopsis embeddings and nonlinear tree ensembles.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

sys.path.append(os.path.abspath('..'))
from src.data import load_and_merge_data, filter_clean_theatrical
from src.features import build_features
from src.metrics import evaluate_model

df_clean = filter_clean_theatrical(load_and_merge_data())
X_pre, _, y = build_features(df_clean, max_tfidf_features=150)

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_pre, y, df_clean.index, test_size=0.2, random_state=42, stratify=y
)
test_slice = df_clean.loc[idx_test]


## 2. Model 2: Random Forest & Hyperparameter Grid Search
Systematically explore tree count, max depth, and split criteria.


In [ ]:
rf_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), rf_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print(f"Best RF CV ROC-AUC: {grid_rf.best_score_:.4f}")
print(f"Best RF Params:     {grid_rf.best_params_}")


## 3. Feature Importance Analysis
Identify the strongest pre-release predictors of theatrical success.


In [ ]:
best_rf = grid_rf.best_estimator_
importances = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

print("Top 15 Most Influential Pre-Release Features:")
print(importances.head(15))


## 4. Comparative Performance Summary
Compare Linear Baseline (V1) against Ensemble Model (V2).


In [ ]:
y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

res_rf = evaluate_model("Random Forest (Tuned)", "Pre-release", y_test, y_pred_rf, y_prob_rf, test_slice)
pd.DataFrame([res_rf])
